In [1]:
import argparse
import os
import pathlib
import sys
import uuid

import duckdb
import pandas as pd
from cytotable import convert, presets
from image_analysis_3D.file_utils.arg_parsing_utils import parse_args
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)
from parsl.config import Config
from parsl.executors import HighThroughputExecutor

root_dir, in_notebook = init_notebook()

profile_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot/NF1_organoid_data")).resolve(),
    root_dir,
)

In [2]:
if not in_notebook:
    args = parse_args()
    well_fov = args["well_fov"]
    patient = args["patient"]
    image_based_profiles_subparent_name = args["image_based_profiles_subparent_name"]

else:
    patient = "NF0014_T1"
    well_fov = "C4-1"
    image_based_profiles_subparent_name = "image_based_profiles"

In [3]:
input_sqlite_file = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/{well_fov}.duckdb"
).resolve(strict=True)
destination_sc_parquet_file = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/sc_profiles_{well_fov}.parquet"
).resolve()
destination_organoid_parquet_file = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/organoid_profiles_{well_fov}.parquet"
).resolve()
destination_nucleocentric_parquet_file = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/nucleocentric_profiles_{well_fov}.parquet"
).resolve()
destination_sc_parquet_file.parent.mkdir(parents=True, exist_ok=True)
dest_datatype = "parquet"

In [4]:
# show the tables
with duckdb.connect(input_sqlite_file) as con:
    tables = con.execute("SHOW TABLES").fetchdf()
    print(tables)
    nuclei_table = con.sql("SELECT * FROM Nuclei").df()
    cells_table = con.sql("SELECT * FROM Cell").df()
    cytoplasm_table = con.sql("SELECT * FROM Cytoplasm").df()
    organoid_table = con.sql("SELECT * FROM Organoid").df()
    nucleocentric_table = con.sql("SELECT * FROM Nucleocentric").df()

            name
0           Cell
1      Cytoplasm
2         Nuclei
3  Nucleocentric
4       Organoid


In [5]:
nuclei_id_set = set(nuclei_table["object_id"].to_list())
cells_id_set = set(cells_table["object_id"].to_list())
cytoplasm_id_set = set(cytoplasm_table["object_id"].to_list())
# find the intersection of the three sets
intersection_set = nuclei_id_set.intersection(cells_id_set, cytoplasm_id_set)
# keep only the rows in the three tables that are in the intersection set
nuclei_table = nuclei_table[nuclei_table["object_id"].isin(intersection_set)]
cells_table = cells_table[cells_table["object_id"].isin(intersection_set)]
cytoplasm_table = cytoplasm_table[cytoplasm_table["object_id"].isin(intersection_set)]

In [6]:
# connect to DuckDB and register the tables
with duckdb.connect() as con:
    con.register("nuclei", nuclei_table)
    con.register("cells", cells_table)
    con.register("cytoplasm", cytoplasm_table)
    # Merge them with SQL
    merged_df = con.execute("""
        SELECT *
        FROM nuclei
        LEFT JOIN cells USING (object_id)
        LEFT JOIN cytoplasm USING (object_id)
    """).df()

## Reorder object IDs

In [7]:
# replace the object_id with a new unique ID
organoid_table["object_id"] = [i for i in range(1, organoid_table.shape[0] + 1)]
merged_df["object_id"] = [i for i in range(1, merged_df.shape[0] + 1)]
nucleocentric_table["object_id"] = [
    i for i in range(1, nucleocentric_table.shape[0] + 1)
]

In [8]:
# save the organoid data as parquet
print(f"Final organoid data shape: {organoid_table.shape}")
organoid_table.to_parquet(destination_organoid_parquet_file, index=False)
organoid_table.head()

Final organoid data shape: (1, 3337)


,object_id,image_set,Organoid_NoChannel_AreaSizeShape_Volume,Organoid_NoChannel_AreaSizeShape_CenterX,Organoid_NoChannel_AreaSizeShape_CenterY,Organoid_NoChannel_AreaSizeShape_CenterZ,Organoid_NoChannel_AreaSizeShape_BboxVolume,Organoid_NoChannel_AreaSizeShape_MinX,Organoid_NoChannel_AreaSizeShape_MaxX,Organoid_NoChannel_AreaSizeShape_MinY,...,Organoid_Mito_Texture_DifferenceEntropy-256-3,Organoid_Mito_Texture_DifferenceVariance-256-3,Organoid_Mito_Texture_Entropy-256-3,Organoid_Mito_Texture_InformationMeasureOfCorrelation1-256-3,Organoid_Mito_Texture_InformationMeasureOfCorrelation2-256-3,Organoid_Mito_Texture_InverseDifferenceMoment-256-3,Organoid_Mito_Texture_SumAverage-256-3,Organoid_Mito_Texture_SumEntropy-256-3,Organoid_Mito_Texture_SumVariance-256-3,Organoid_Mito_Texture_Variance-256-3
0,1,C4-1,27421038.0,711.460877,936.634612,19.864556,50391450.0,267,1082,109,...,3.316532,0.000513,6.680345,-0.057098,0.564855,0.228559,71.61064,4.750468,104.779099,35.706212


In [9]:
print(f"Final merged single cell dataframe shape: {merged_df.shape}")
# save the sc data as parquet
merged_df.to_parquet(destination_sc_parquet_file, index=False)
merged_df.head()

Final merged single cell dataframe shape: (55, 10011)


,object_id,image_set,Nuclei_NoChannel_AreaSizeShape_Volume,Nuclei_NoChannel_AreaSizeShape_CenterX,Nuclei_NoChannel_AreaSizeShape_CenterY,Nuclei_NoChannel_AreaSizeShape_CenterZ,Nuclei_NoChannel_AreaSizeShape_BboxVolume,Nuclei_NoChannel_AreaSizeShape_MinX,Nuclei_NoChannel_AreaSizeShape_MaxX,Nuclei_NoChannel_AreaSizeShape_MinY,...,Cytoplasm_ER_Texture_DifferenceEntropy-256-3,Cytoplasm_ER_Texture_DifferenceVariance-256-3,Cytoplasm_ER_Texture_Entropy-256-3,Cytoplasm_ER_Texture_InformationMeasureOfCorrelation1-256-3,Cytoplasm_ER_Texture_InformationMeasureOfCorrelation2-256-3,Cytoplasm_ER_Texture_InverseDifferenceMoment-256-3,Cytoplasm_ER_Texture_SumAverage-256-3,Cytoplasm_ER_Texture_SumEntropy-256-3,Cytoplasm_ER_Texture_SumVariance-256-3,Cytoplasm_ER_Texture_Variance-256-3
0,1,C4-1,7480.0,881.128610,443.802139,1.488770,10200.0,857,907,418,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,C4-1,37965.0,570.619781,889.462874,4.402107,53088.0,533,612,848,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,C4-1,45736.0,630.840716,980.077226,4.681083,80442.0,576,685,939,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,C4-1,32804.0,513.994940,1233.819229,3.297586,55692.0,471,562,1178,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,C4-1,78836.0,805.882478,657.815478,9.888135,139956.0,752,859,604,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
print(f"Final nucleocentric dataframe shape: {nucleocentric_table.shape}")
# save the nucleocentric data as parquet
nucleocentric_table.to_parquet(destination_nucleocentric_parquet_file, index=False)
nucleocentric_table.head()

Final nucleocentric dataframe shape: (55, 3074)


,object_id,image_set,Nucleocentric_ER_CHAMMI75_Feature0,Nucleocentric_ER_CHAMMI75_Feature1,Nucleocentric_ER_CHAMMI75_Feature10,Nucleocentric_ER_CHAMMI75_Feature100,Nucleocentric_ER_CHAMMI75_Feature101,Nucleocentric_ER_CHAMMI75_Feature102,Nucleocentric_ER_CHAMMI75_Feature103,Nucleocentric_ER_CHAMMI75_Feature104,...,Nucleocentric_DNA_SAMMed3D_Feature90,Nucleocentric_DNA_SAMMed3D_Feature91,Nucleocentric_DNA_SAMMed3D_Feature92,Nucleocentric_DNA_SAMMed3D_Feature93,Nucleocentric_DNA_SAMMed3D_Feature94,Nucleocentric_DNA_SAMMed3D_Feature95,Nucleocentric_DNA_SAMMed3D_Feature96,Nucleocentric_DNA_SAMMed3D_Feature97,Nucleocentric_DNA_SAMMed3D_Feature98,Nucleocentric_DNA_SAMMed3D_Feature99
0,1,C4-1,-2.034619,-3.223986,5.187386,0.839193,-4.609393,-1.314024,2.809356,0.652013,...,-0.006334,-0.050433,0.088730,-0.010508,0.028697,0.023633,-0.013850,0.238845,0.333358,0.229721
1,2,C4-1,-2.077653,-3.769540,7.392309,-2.904451,0.304836,-0.125569,3.759428,1.653862,...,-0.007324,-0.063565,0.012365,-0.010451,0.018819,-0.035347,-0.072271,0.225065,0.375739,0.230366
2,3,C4-1,1.937732,-3.106618,3.096840,2.169843,-0.647295,-3.975330,2.363535,-1.013884,...,-0.008185,-0.074491,-0.133589,-0.010789,0.014343,-0.093814,0.208879,0.276905,0.316322,0.034441
3,4,C4-1,0.399118,-2.973527,2.263089,2.779028,0.539425,-0.112796,4.970295,-2.148762,...,-0.006136,-0.088927,0.062739,-0.010761,0.023554,0.038025,0.025290,0.201641,0.315303,0.227351
4,5,C4-1,-2.808134,-5.294312,0.994752,2.960990,-2.145890,1.459159,6.289212,-0.280198,...,-0.007903,-0.036514,0.055145,-0.010306,0.020019,-0.006451,-0.026692,0.222935,0.391211,0.207672
